In [16]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [17]:
# ============================================================================
# 1. Data Loading
# ============================================================================
print("\n[Step 1] Loading datasets...")

try:
    # Load SalesForce dataset
    df1 = pd.read_csv('dataset1.csv')
    print(f"✓ Dataset1 (SalesForce) loaded: {len(df1)} rows, {len(df1.columns)} columns")
    
    # Load usage logs datasets
    df2a = pd.read_csv('dataset2a.csv')
    print(f"✓ Dataset2a (Q1-Q2 logs) loaded: {len(df2a)} rows")
    
    df2b = pd.read_csv('dataset2b.csv')
    print(f"✓ Dataset2b (Q3-Q4 logs) loaded: {len(df2b)} rows")
    
    # Load support tickets dataset
    df3 = pd.read_csv('dataset3.csv')
    print(f"✓ Dataset3 (support tickets) loaded: {len(df3)} rows")
    
except Exception as e:
    print(f"✗ Data loading failed: {e}")
    raise



[Step 1] Loading datasets...
✓ Dataset1 (SalesForce) loaded: 3000 rows, 17 columns
✓ Dataset2a (Q1-Q2 logs) loaded: 546000 rows
✓ Dataset2b (Q3-Q4 logs) loaded: 552000 rows
✓ Dataset3 (support tickets) loaded: 6429 rows


In [18]:
# ============================================================================
# 2. Dataset1 Cleaning - SalesForce data
# ============================================================================
print("\n[Step 2] Cleaning Dataset1 (SalesForce data)...")

# 2.1 Data quality checks
print("\n--- Dataset1 data quality check ---")
print(f"Total rows: {len(df1)}")
print(f"Total columns: {len(df1.columns)}")
print(f"\nMissing values summary:")
missing_df1 = df1.isnull().sum()
missing_df1 = missing_df1[missing_df1 > 0].sort_values(ascending=False)
if len(missing_df1) > 0:
    for col, count in missing_df1.items():
        pct = (count / len(df1)) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")
else:
    print("  No missing values")

# 2.2 Handle date fields
print("\n--- Handling date fields ---")
df1['contract_start_date'] = pd.to_datetime(df1['contract_start_date'], format='%m/%d/%Y', errors='coerce')
df1['contract_end_date'] = pd.to_datetime(df1['contract_end_date'], format='%m/%d/%Y', errors='coerce')
print(f"✓ Date fields converted")

# 2.3 Handle missing industry field (~30% missing)
print("\n--- Handling missing industry field ---")
industry_missing_before = df1['industry'].isnull().sum()
industry_missing_pct = (industry_missing_before / len(df1)) * 100
print(f"Missing count: {industry_missing_before} ({industry_missing_pct:.1f}%)")

# Strategy: mark missing industry as 'Unknown'
df1['industry'] = df1['industry'].fillna('Unknown')
print(f"✓ Filled missing industry with 'Unknown'")

# 2.4 Data consistency checks
print("\n--- Data consistency check ---")
print(f"Unique customers: {df1['customer_id'].nunique()}")
print(f"Total records: {len(df1)}")
if df1['customer_id'].nunique() != len(df1):
    print("⚠️ Warning: duplicate customer_id found")
    duplicates = df1[df1.duplicated(subset=['customer_id'], keep=False)]
    print(f"Duplicate records: {len(duplicates)}")
else:
    print("✓ No duplicate customer_id")

# 2.5 Churn checks
print("\n--- Churn checks ---")
churned_count = df1['is_churned'].sum()
churned_pct = (churned_count / len(df1)) * 100
print(f"Churned customers: {churned_count} ({churned_pct:.1f}%)")
print(f"Retained customers: {len(df1) - churned_count} ({100 - churned_pct:.1f}%)")

# 2.6 Save cleaned Dataset1
df1_cleaned = df1.copy()
df1_cleaned.to_csv('dataset1_cleaned.csv', index=False)
print(f"\n✓ Dataset1 cleaning completed, saved as 'dataset1_cleaned.csv'")



[Step 2] Cleaning Dataset1 (SalesForce data)...

--- Dataset1 data quality check ---
Total rows: 3000
Total columns: 17

Missing values summary:
  contract_end_date: 1428 (47.6%)
  industry: 862 (28.7%)

--- Handling date fields ---
✓ Date fields converted

--- Handling missing industry field ---
Missing count: 862 (28.7%)
✓ Filled missing industry with 'Unknown'

--- Data consistency check ---
Unique customers: 3000
Total records: 3000
✓ No duplicate customer_id

--- Churn checks ---
Churned customers: 364 (12.1%)
Retained customers: 2636 (87.9%)

✓ Dataset1 cleaning completed, saved as 'dataset1_cleaned.csv'


In [19]:
# ============================================================================
# 3. Dataset2a & 2b Cleaning - Usage logs
# ============================================================================
print("\n[Step 3] Cleaning Dataset2a & 2b (usage logs)...")

# 3.1 Merge the two log datasets
print("\n--- Merging logs ---")
df2a['date'] = pd.to_datetime(df2a['date'])
df2b['date'] = pd.to_datetime(df2b['date'])
print(f"Dataset2a date range: {df2a['date'].min()} to {df2a['date'].max()}")
print(f"Dataset2b date range: {df2b['date'].min()} to {df2b['date'].max()}")

# Merge the two datasets
df2_combined = pd.concat([df2a, df2b], ignore_index=True)
print(f"✓ Combined records: {len(df2_combined)}")

# 3.2 Handle corrupted EU September logs
print("\n--- Handling corrupted EU September logs ---")
# First get the list of EU customers from dataset1
eu_customers = set(df1[df1['is_eu'] == 1]['customer_id'].unique())
print(f"Total EU customers: {len(eu_customers)}")

# Identify EU customer logs in September
september_2024 = pd.Timestamp('2024-09-01')
october_2024 = pd.Timestamp('2024-10-01')
september_eu_logs = df2_combined[
    (df2_combined['customer_id'].isin(eu_customers)) &
    (df2_combined['date'] >= september_2024) &
    (df2_combined['date'] < october_2024)
]
print(f"September EU logs count: {len(september_eu_logs)}")

# Mark corrupted data (do not delete; keep for analysis)
df2_combined['is_corrupted'] = (
    (df2_combined['customer_id'].isin(eu_customers)) &
    (df2_combined['date'] >= september_2024) &
    (df2_combined['date'] < october_2024)
)
corrupted_count = df2_combined['is_corrupted'].sum()
print(f"✓ Marked {corrupted_count} corrupted records (September EU logs)")

# 3.3 Data quality checks for logs
print("\n--- Log data quality check ---")
print(f"Missing values summary:")
missing_df2 = df2_combined.isnull().sum()
missing_df2 = missing_df2[missing_df2 > 0]
if len(missing_df2) > 0:
    for col, count in missing_df2.items():
        print(f"  {col}: {count}")
else:
    print("  No missing values")

# Check anomalies
print(f"\nAnomaly checks:")
print(f"  Negative logins: {(df2_combined['logins'] < 0).sum()}")
print(f"  Negative session minutes: {(df2_combined['session_minutes'] < 0).sum()}")

# 3.4 Save cleaned logs
df2_cleaned = df2_combined.copy()
df2_cleaned.to_csv('dataset2_cleaned.csv', index=False)
print(f"\n✓ Dataset2 cleaned and saved to 'dataset2_cleaned.csv'")



[Step 3] Cleaning Dataset2a & 2b (usage logs)...

--- Merging logs ---
Dataset2a date range: 2024-01-01 00:00:00 to 2024-06-30 00:00:00
Dataset2b date range: 2024-07-01 00:00:00 to 2024-12-31 00:00:00
✓ Combined records: 1098000

--- Handling corrupted EU September logs ---
Total EU customers: 1867
September EU logs count: 56010
✓ Marked 56010 corrupted records (September EU logs)

--- Log data quality check ---
Missing values summary:
  logins: 56010
  feature_events: 56010
  session_minutes: 56010

Anomaly checks:
  Negative logins: 0
  Negative session minutes: 0

✓ Dataset2 cleaned and saved to 'dataset2_cleaned.csv'


In [20]:
# ============================================================================
# 4. Dataset3 Cleaning - Support tickets
# ============================================================================
print("\n[Step 4] Cleaning Dataset3 (support tickets)...")

# 4.1 Handle datetime fields
print("\n--- Handling datetime fields ---")
df3['created_at'] = pd.to_datetime(df3['created_at'], format='%Y-%m-%dT%H:%M', errors='coerce')
print(f"✓ Datetime fields converted")
print(f"Ticket time range: {df3['created_at'].min()} to {df3['created_at'].max()}")

# 4.2 Data quality checks for tickets
print("\n--- Ticket data quality check ---")
print(f"Total tickets: {len(df3)}")
print(f"Unique customers: {df3['customer_id'].nunique()}")
print(f"Missing values summary:")
missing_df3 = df3.isnull().sum()
missing_df3 = missing_df3[missing_df3 > 0]
if len(missing_df3) > 0:
    for col, count in missing_df3.items():
        print(f"  {col}: {count}")
else:
    print("  No missing values")

# 4.3 Check issue category distribution
print(f"\nIssue category distribution:")
print(df3['issue_category'].value_counts())

# 4.4 Save cleaned ticket data
df3_cleaned = df3.copy()
df3_cleaned.to_csv('dataset3_cleaned.csv', index=False)
print(f"\n✓ Dataset3 cleaned and saved to 'dataset3_cleaned.csv'")



[Step 4] Cleaning Dataset3 (support tickets)...

--- Handling datetime fields ---
✓ Datetime fields converted
Ticket time range: 2024-01-05 15:12:00 to 2024-12-30 23:28:00

--- Ticket data quality check ---
Total tickets: 6429
Unique customers: 2421
Missing values summary:
  No missing values

Issue category distribution:
issue_category
billing_admin          1671
product_usability      1622
product_performance    1592
sales_expectation      1544
Name: count, dtype: int64

✓ Dataset3 cleaned and saved to 'dataset3_cleaned.csv'


In [21]:
# ============================================================================
# 5. Data merging - based on customer_id
# ============================================================================
print("\n[Step 5] Merging data (based on customer_id)...")

# 5.1 Check customer_id consistency
print("\n--- Checking customer_id consistency ---")
customers_df1 = set(df1['customer_id'].unique())
customers_df2 = set(df2_combined['customer_id'].unique())
customers_df3 = set(df3['customer_id'].unique())

print(f"Dataset1 unique customers: {len(customers_df1)}")
print(f"Dataset2 unique customers: {len(customers_df2)}")
print(f"Dataset3 unique customers: {len(customers_df3)}")

# Find customers present in all datasets
all_customers = customers_df1 & customers_df2 & customers_df3
print(f"\nCustomers present in all datasets: {len(all_customers)}")

# Find customers only in one dataset
only_df1 = customers_df1 - customers_df2 - customers_df3
only_df2 = customers_df2 - customers_df1 - customers_df3
only_df3 = customers_df3 - customers_df1 - customers_df2
print(f"Customers only in Dataset1: {len(only_df1)}")
print(f"Customers only in Dataset2: {len(only_df2)}")
print(f"Customers only in Dataset3: {len(only_df3)}")

# 5.2 Create merged dataset (left join on Dataset1)
print("\n--- Creating merged dataset ---")
df_merged = df1_cleaned.copy()

# Compute customer-level aggregates from logs
print("  Computing usage aggregates...")
df2_agg = df2_combined.groupby('customer_id').agg({
    'logins': ['sum', 'mean', 'count'],
    'feature_events': ['sum', 'mean'],
    'session_minutes': ['sum', 'mean']
}).reset_index()
df2_agg.columns = ['customer_id', 
                   'total_logins', 'avg_daily_logins', 'active_days',
                   'total_feature_events', 'avg_feature_events',
                   'total_session_minutes', 'avg_session_minutes']

# Compute Q1-Q2 and Q3-Q4 metrics separately
df2a_agg = df2a.groupby('customer_id').agg({
    'logins': 'sum',
    'session_minutes': 'sum'
}).reset_index()
df2a_agg.columns = ['customer_id', 'q1q2_logins', 'q1q2_session_minutes']

df2b_agg = df2b.groupby('customer_id').agg({
    'logins': 'sum',
    'session_minutes': 'sum'
}).reset_index()
df2b_agg.columns = ['customer_id', 'q3q4_logins', 'q3q4_session_minutes']

# Merge usage metrics
df_merged = df_merged.merge(df2_agg, on='customer_id', how='left')
df_merged = df_merged.merge(df2a_agg, on='customer_id', how='left')
df_merged = df_merged.merge(df2b_agg, on='customer_id', how='left')

# Compute customer-level aggregates from tickets
print("  Computing ticket aggregates...")
df3_agg = df3.groupby('customer_id').agg({
    'ticket_id': 'count',
    'issue_category': lambda x: x.value_counts().to_dict(),
    'sentiment': 'mean',
    'first_response_hours': 'mean',
    'resolution_hours': 'mean',
    'resolved': 'sum'
}).reset_index()
df3_agg.columns = ['customer_id', 'total_tickets', 'issue_category_dist', 
                   'avg_sentiment', 'avg_first_response_hours', 
                   'avg_resolution_hours', 'resolved_tickets']

# Compute ticket counts per category
issue_categories = df3['issue_category'].unique()
for category in issue_categories:
    category_tickets = df3[df3['issue_category'] == category].groupby('customer_id')['ticket_id'].count().reset_index()
    category_tickets.columns = ['customer_id', f'tickets_{category}']
    df_merged = df_merged.merge(category_tickets, on='customer_id', how='left')
    df_merged[f'tickets_{category}'] = df_merged[f'tickets_{category}'].fillna(0)

# Merge ticket aggregates
df_merged = df_merged.merge(df3_agg[['customer_id', 'total_tickets', 'avg_sentiment', 
                                     'avg_first_response_hours', 'avg_resolution_hours', 
                                     'resolved_tickets']], 
                           on='customer_id', how='left')
df_merged['total_tickets'] = df_merged['total_tickets'].fillna(0)
df_merged['resolved_tickets'] = df_merged['resolved_tickets'].fillna(0)

print(f"✓ Data merge completed")
print(f"Merged dataset: {len(df_merged)} rows, {len(df_merged.columns)} columns")

# Save merged dataset
df_merged.to_csv('dataset_merged.csv', index=False)
print(f"✓ Saved to 'dataset_merged.csv'")



[Step 5] Merging data (based on customer_id)...

--- Checking customer_id consistency ---
Dataset1 unique customers: 3000
Dataset2 unique customers: 3000
Dataset3 unique customers: 2421

Customers present in all datasets: 2421
Customers only in Dataset1: 0
Customers only in Dataset2: 0
Customers only in Dataset3: 0

--- Creating merged dataset ---
  Computing usage aggregates...
  Computing ticket aggregates...
✓ Data merge completed
Merged dataset: 3000 rows, 37 columns
✓ Saved to 'dataset_merged.csv'


In [22]:
print("=" * 60)
print("Refining merged data - filling missing values")
print("=" * 60)

# Load merged data
df_merged = pd.read_csv('dataset_merged.csv')
print(f"\nOriginal data: {len(df_merged)} rows, {len(df_merged.columns)} columns")

# Check missing values
print("\n--- Missing values check ---")
missing_before = df_merged.isnull().sum()
missing_cols = missing_before[missing_before > 0]
if len(missing_cols) > 0:
    print("Missing values summary:")
    for col, count in missing_cols.items():
        pct = (count / len(df_merged)) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")
else:
    print("  No missing values")

# Fill missing usage metrics
print("\n--- Filling missing values for usage metrics ---")
usage_cols = [
    'total_logins', 'avg_daily_logins', 'active_days',
    'total_feature_events', 'avg_feature_events',
    'total_session_minutes', 'avg_session_minutes',
    'q1q2_logins', 'q1q2_session_minutes',
    'q3q4_logins', 'q3q4_session_minutes',
]

# Check which columns exist
existing_cols = [col for col in usage_cols if col in df_merged.columns]
missing_in_cols = df_merged[existing_cols].isnull().sum().sum()

if missing_in_cols > 0:
    df_merged[existing_cols] = df_merged[existing_cols].fillna(0)
    print(f"✓ Filled {missing_in_cols} missing values")
else:
    print("✓ No missing usage values to fill")

# Fill missing ticket-related metrics (if present)
print("\n--- Filling missing values for ticket metrics ---")
ticket_cols = [
    'avg_sentiment', 'avg_first_response_hours', 
    'avg_resolution_hours',
]

existing_ticket_cols = [col for col in ticket_cols if col in df_merged.columns]
missing_in_ticket = df_merged[existing_ticket_cols].isnull().sum().sum()

if missing_in_ticket > 0:
    # For customers without tickets, fill sentiment and response times with 0
    df_merged[existing_ticket_cols] = df_merged[existing_ticket_cols].fillna(0)
    print(f"✓ Filled {missing_in_ticket} missing values")
else:
    print("✓ No missing ticket values to fill")

# Data validation
print("\n--- Data validation ---")
print(f"Total missing values: {df_merged.isnull().sum().sum()}")
print(f"Negative logins: {(df_merged['total_logins'] < 0).sum() if 'total_logins' in df_merged.columns else 'N/A'}")
print(f"Negative session minutes: {(df_merged['total_session_minutes'] < 0).sum() if 'total_session_minutes' in df_merged.columns else 'N/A'}")

# Save refined data
output_file = 'dataset_merged_final.csv'
df_merged.to_csv(output_file, index=False)
print(f"\n✓ Refined data saved to '{output_file}'")
print(f"  Total rows: {len(df_merged)}")
print(f"  Total columns: {len(df_merged.columns)}")
print(f"  Missing values: {df_merged.isnull().sum().sum()}")

print("\n" + "=" * 60)
print("Recommendations:")
print("  - Use 'dataset_merged_final.csv' for downstream analysis")
print("  - Keep the original 'dataset_merged.csv' as a backup")
print("=" * 60)
print("\nData refinement complete!")
print("=" * 60)

Refining merged data - filling missing values

Original data: 3000 rows, 37 columns

--- Missing values check ---
Missing values summary:
  contract_end_date: 1428 (47.6%)
  avg_sentiment: 579 (19.3%)
  avg_first_response_hours: 579 (19.3%)
  avg_resolution_hours: 579 (19.3%)

--- Filling missing values for usage metrics ---
✓ No missing usage values to fill

--- Filling missing values for ticket metrics ---
✓ Filled 1737 missing values

--- Data validation ---
Total missing values: 1428
Negative logins: 0
Negative session minutes: 0

✓ Refined data saved to 'dataset_merged_final.csv'
  Total rows: 3000
  Total columns: 37
  Missing values: 1428

Recommendations:
  - Use 'dataset_merged_final.csv' for downstream analysis
  - Keep the original 'dataset_merged.csv' as a backup

Data refinement complete!
